# 07 · Fine-Tuning, PEFT & LoRA

> **Source notes:** `FineTuning.md`

When is fine-tuning the right call? This notebook covers:
- **Decision tree** — should you fine-tune at all?
- **LoRA math visualised** — low-rank decomposition and parameter savings
- **Dataset preparation** — building and validating a fine-tuning dataset
- **PEFT config** — ready-to-adapt LoRA training setup
- **Distillation workflow** — generating synthetic data from a larger model

> GPU recommended for actual training. Math and dataset cells run on CPU.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `run()` to produce the result
#
# Hint:
#    # implement using the APIs described above

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','numpy','matplotlib','transformers','peft','datasets','ollama','-q'],check=True)
print('Packages ready.')
import numpy as np, matplotlib.pyplot as plt, json, re

## 1 · The Fine-Tuning Decision Tree

```
Failing output format? -> Prompt engineering / structured output
Missing domain facts? -> RAG (fine-tuning memorises facts poorly)
Style/behaviour failure? -> Fine-tuning v
Too slow/expensive? -> Distillation v
Proprietary syntax/DSL? -> Fine-tuning v
Reasoning failures? -> Better model or CoT
```

In [ ]:
def fine_tune_decision(symptoms):
    """
    TODO #2: Implement `fine_tune_decision()`.

    Steps:
    1. Define helper function `fine_tune_decision()`
    2. Compute `scenarios`

    Hint:
    # implement using the APIs described above

    Returns: 'NEED MORE INFO -- describe the failu...
    """
    raise NotImplementedError("TODO: implement fine_tune_decision()")

## 2 · LoRA Math — Low-Rank Decomposition

LoRA adds: **W' = W + BA** where A in R(r x k), B in R(d x r), r << min(d,k).
Trains r(d+k) params instead of d*k — typically 0.1-3% of full params.

In [ ]:
def lora_stats(d, k, r):
    """
    TODO #3: Implement `lora_stats()`.

    Steps:
    1. Define helper function `lora_stats()`
    2. Compute `configs`
    3. Plot results -- call `subplots()`

    Hint:
    ax = plt.subplots(???)

    Returns: {'full': full, 'lora': lora, 'pct': l...
    """
    raise NotImplementedError("TODO: implement lora_stats()")

## 3 · Dataset Preparation & Validation

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `TRAINING_DATA` using `dumps()`
#
# Hint:
#    # implement using the APIs described above

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
TRAINING_DATA = [
    {'system': 'Parse orders to JSON: {items:[{name,size,qty,modifiers}],delivery_address,note}',
     'user':   'Two large Margheritas, extra cheese, to 42 Maple Street.',
     'assistant': json.dumps({'items':[{'name':'Margherita','size':'large','qty':2,'modifiers':['extra cheese']}],'delivery_address':'42 Maple Street','note':None})},
    {'system': 'Parse orders to JSON: {items:[{name,size,qty,modifiers}],delivery_address,note}',
     'user':   'Small Pepperoni for collection, gluten-free base please.',
     'assistant': json.dumps({'items':[{'name':'Pepperoni','size':'small','qty':1,'modifiers':['gluten-free base']}],'delivery_address':None,'note':'collection'})},
    {'system': 'Parse orders to JSON: {items:[{name,size,qty,modifiers}],delivery_address,note}',
     'user':   'What time do you close?',
     'assistant': json.dumps({'error':'not_an_order','message':'This does not appear to be a pizza order.'})},
]
print('Dataset validation:')
errors = 0
for i, row in enumerate(TRAINING_DATA):
    try:
        json.loads(row['assistant']); print(f'  [{i+1}] OK  {row["user"][:55]}')
    except json.JSONDecodeError as e:
        print(f'  [{i+1}] ERR {e}'); errors += 1
print(f'\n{len(TRAINING_DATA)-errors}/{len(TRAINING_DATA)} valid.')
with open('ft_dataset.jsonl','w') as f:
    [f.write(json.dumps(r)+'\n') for r in TRAINING_DATA]
print('Exported ft_dataset.jsonl')

## 4 · PEFT / LoRA Config

Key hyperparameters:
- `r=16` — rank (default; lower=fewer params, higher=more capacity)
- `lora_alpha=32` — effective scale = alpha/r = 2.0
- `target_modules` — which matrices to adapt (Q and V projections for attention)
- `lora_dropout=0.05` — regularisation

Actual training requires GPU with ~16 GB VRAM for a 7B model.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create visualisation
# 2. Compute `lora_cfg` using `LoraConfig()`
#
# Hint:
#    lora_cfg = LoraConfig(task_type=???, inference_mode=???)
#    train_args = TrainingArguments(output_dir=???, num_train_epochs=???)
#    model = AutoModelForCausalLM.from_pretrained(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
from peft import LoraConfig, TaskType
from transformers import TrainingArguments

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM, inference_mode=False,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','v_proj'], bias='none',
)
train_args = TrainingArguments(
    output_dir='./ft-lora-out', num_train_epochs=3,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-4, fp16=True, logging_steps=10,
    save_strategy='epoch', warmup_ratio=0.05, lr_scheduler_type='cosine', report_to='none',
)
d = 4096
trainable = lora_cfg.r * (d + d) * 2  # Q+V for one layer
full = d * d * 2
print('LoRA Config:')
print(f'  r={lora_cfg.r}  alpha={lora_cfg.lora_alpha}  scale={lora_cfg.lora_alpha/lora_cfg.r:.1f}')
print(f'  Trainable per layer: {trainable:,} ({trainable/full:.2%} of full)')
print('\nTo train:')
print('  model = AutoModelForCausalLM.from_pretrained(...)')
print('  model = get_peft_model(model, lora_cfg)')
print('  # then use SFTTrainer from trl or standard Trainer')

## 5 · Distillation — Synthetic Training Data from Teacher

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `MODEL` using `chat()`
#
# Hint:
#    resp = ollama.chat(???)
#    m = re.search(???)
#    ex = json.loads(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
import ollama
MODEL = 'phi3:mini'
TEACHER_SYS = 'Generate a realistic pizza order variant and its correct JSON parse. Respond with valid JSON only: {"input": "<order text>", "output": {"items": [...], "delivery_address": ..., "note": ...}}'
SEEDS = [
    'A large Margherita and two Garlic Breads to 5 Elm Road.',
    'One medium Pepperoni, no onions, for collection.',
    'Three small Veggie Supremes to Flat 4, 12 Oak St -- leave at door.',
]
dataset = []
print('Generating synthetic training data from teacher model...\n')
for seed in SEEDS:
    resp = ollama.chat(model=MODEL, messages=[{'role':'system','content':TEACHER_SYS},{'role':'user','content':f'Seed: {seed}'}], options={'temperature':0.8,'num_predict':200})
    raw = resp['message']['content'].strip()
    m   = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try:
            ex = json.loads(m.group()); dataset.append(ex)
            print(f'  OK  {str(ex.get("input",""))[:65]}')
        except: print(f'  FAIL parse -- seed: {seed[:50]}')
    else: print(f'  FAIL no JSON -- seed: {seed[:50]}')
print(f'\nGenerated {len(dataset)} examples. Scale to 500-5000 for a real distillation run.')

## 6 · DPO — Direct Preference Optimisation

After LoRA fine-tunes *what* the model says, DPO fine-tunes *what* the model **prefers**.

### Training Data Format

```python
{
 "prompt": "What pizza do you recommend?",
 "chosen": "Oh, you've gotta try the Margherita — Nonna's recipe, flying out the door!", # y_w
 "rejected": "The Margherita pizza is available." # y_l
}
```

### DPO Loss

$$\mathcal{L}_{DPO} = -\mathbb{E}\!\left[\log\sigma\!\left(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right]$$

| Symbol | Meaning |
|---|---|
| $\pi_\theta$ | Trainable policy (your LoRA model, being updated) |
| $\pi_{ref}$ | Frozen reference (same model before DPO — never updated) |
| $\beta$ | Temperature: controls KL regularisation (start at 0.1) |
| $y_w / y_l$ | Preferred (warm voice) / rejected (flat response) |

### When to Use

| Signal | Action |
|---|---|
| Brand voice 95%+ but 5% still cold/generic | Add DPO on top of LoRA |
| Style still wrong (flat, incorrect format) | Fix LoRA first — DPO refines, doesn't correct |
| No preference pairs available | LoRA/SFT only — DPO requires (chosen, rejected) |

**PizzaBot impact:** LoRA → 95% warm responses; LoRA + DPO → 99%+ → AOV: $41.00 → $42.50 (+3.7%)

In [ ]:
# TODO: Implement DPO training with TRL DPOTrainer
#
# Steps:
# 1. Build a small preference dataset (at least 5 examples):
#    Each example needs: "prompt", "chosen" (warm Mamma Rosa voice), "rejected" (flat voice)
#    preference_data = [
#        {"prompt": "...", "chosen": "Oh, you've gotta try...", "rejected": "The pizza is available."},
#        ...
#    ]
#
# 2. Configure DPO:
#    from trl import DPOTrainer, DPOConfig
#    dpo_config = DPOConfig(
#        beta=0.1,           # KL regularisation strength
#        learning_rate=5e-7, # Lower than LoRA
#        num_train_epochs=1, # Typically 1 epoch for DPO
#        output_dir="./dpo-pizzabot",
#    )
#
# 3. Initialize DPOTrainer:
#    dpo_trainer = DPOTrainer(
#        model=lora_model,    # Your LoRA-tuned policy (will be updated)
#        ref_model=ref_model, # Frozen reference model (NOT updated)
#        args=dpo_config,
#        train_dataset=preference_dataset,
#        tokenizer=tokenizer,
#    )
#
# 4. Train and observe metrics:
#    dpo_trainer.train()
#    # Watch: rewards/chosen ↑, rewards/rejected ↓, rewards/margins ↑
#
# 5. Print final metrics and compare brand voice score before vs after DPO

#
# Hint:
#   from trl import DPOTrainer, DPOConfig
#   dpo_config = DPOConfig(beta=0.1, learning_rate=5e-7, num_train_epochs=1, ...)
#   dpo_trainer = DPOTrainer(model=lora_model, ref_model=ref_model, args=dpo_config, ...)
#   dpo_trainer.train()
raise NotImplementedError("Implement DPO training — see ch10 §5.5 in fine-tuning.md")

## Summary

| Concept | Key Takeaway |
|---|---|
| Decision tree | Most problems → prompting or RAG first; fine-tune last |
| LoRA math | r(d+k) params vs d*k full — ~1% at r=16 |
| Data > rank | Data quality beats hyperparameter tuning |
| Distillation | Teacher-generated data is most scalable path |
| DPO | Preference triples (x, y_w, y_l) align what the model prefers — no reward model needed |
| Eval gate | Run evaluation metrics before/after to measure actual gain |

**Next:** [SafetyAndHallucination/notebook.ipynb](../SafetyAndHallucination/notebook.ipynb)